# Notebook 1: Introduction to PyTorch
**Artificial Intelligence · Week 1 · Institute of Technology of Cambodia**

**PyTorch** is the deep learning library we use all semester. Everything in it is built on one object, the **tensor**: a multi-dimensional array, like a NumPy array, that can also run on a GPU. Today is about tensors only: how to create them, compute with them, change their shape, and move them between NumPy, the CPU and the GPU.

**How this notebook works**
1. **Follow along (Parts 0–11).** We go through the notebook together. Type the code yourself, or run the cells, and **read every output**. Change the numbers and run again: that is the fastest way to learn.
2. **Exercises (at the end).** On your own. Each exercise has an empty cell; where the answer is deterministic, the expected output is shown so you can compare.
3. **Deliverable:** the completed notebook, with all exercises, submitted at the end of the session.

| Part | Topic |
|---|---|
| 0 | Setup |
| 1 | What is a tensor? Scalars, vectors, matrices |
| 2 | Creating tensors |
| 3 | Data types |
| 4 | Operations and matrix multiplication |
| 5 | Aggregation: min, max, mean, sum, argmax |
| 6 | Reshaping, stacking, squeezing, permuting |
| 7 | Indexing |
| 8 | Broadcasting, in NumPy and PyTorch |
| 9 | Copying tensors, and PyTorch ↔ NumPy |
| 10 | Reproducibility |
| 11 | Running on the GPU |
| ★ | Exercises |

**Keep the documentation open:** [pytorch.org/docs/stable/tensors.html](https://pytorch.org/docs/stable/tensors.html). Reading documentation is part of the job.

## 0 · Setup
In Colab: **Runtime → Change runtime type → T4 GPU → Save**. Then:

In [1]:
import torch
import numpy as np

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.14.1
GPU available: False


## 1 · What is a tensor?
A tensor is a way to represent numbers. Its **number of dimensions** (`ndim`) is the number of indices you need to reach one number.

*Adapted from D. Bourke, [Learn PyTorch for Deep Learning: 00. PyTorch Fundamentals](https://www.learnpytorch.io/00_pytorch_fundamentals/), MIT License (modified).*

In [2]:
scalar = torch.tensor(7)
print(scalar, "| ndim:", scalar.ndim)
print(scalar.item())            # a one-element tensor back to a Python number

tensor(7) | ndim: 0
7


In [3]:
vector = torch.tensor([7, 7])
print(vector, "| ndim:", vector.ndim, "| shape:", vector.shape)

tensor([7, 7]) | ndim: 1 | shape: torch.Size([2])


In [4]:
matrix = torch.tensor([[7, 8],
                       [9, 10]])
print(matrix, "| ndim:", matrix.ndim, "| shape:", matrix.shape)

tensor([[ 7,  8],
        [ 9, 10]]) | ndim: 2 | shape: torch.Size([2, 2])


In [5]:
tensor = torch.tensor([[[1, 2, 3],
                        [3, 6, 9],
                        [2, 4, 5]]])
print(tensor, "| ndim:", tensor.ndim, "| shape:", tensor.shape)

tensor([[[1, 2, 3],
         [3, 6, 9],
         [2, 4, 5]]]) | ndim: 3 | shape: torch.Size([1, 3, 3])


Count the opening brackets `[` to get `ndim`. The shape lists the size of each dimension, from the outside in: `tensor` has 1 block of 3 rows of 3 numbers.

| Name | ndim | Example shape | In this course |
|---|---|---|---|
| scalar | 0 | `()` | a loss value |
| vector | 1 | `(n,)` | one example with n features; the labels of a batch |
| matrix | 2 | `(m, n)` | a dataset: **m examples in rows, n features in columns** |
| tensor | 3+ | `(N, C, H, W)` | a batch of N images, C color channels, H × W pixels |

**The row convention.** In this course a dataset is always stored with **one example per row**: shape (m, n). This is what PyTorch expects. (Andrew Ng's videos put examples in columns: be careful when you compare.)

## 2 · Creating tensors

### 2.1 From data
`torch.tensor()` copies data from a Python list (or tuple, or NumPy array). Nested lists give more dimensions.

*Adapted from the PyTorch tutorial [Introduction to PyTorch Tensors](https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html), BSD 3-Clause License (modified).*

In [6]:
some_constants = torch.tensor([[3.1415926, 2.71828], [1.61803, 0.0072897]])
print(some_constants)

some_integers = torch.tensor((2, 3, 5, 7, 11, 13, 17, 19))
print(some_integers)

tensor([[3.1416, 2.7183],
        [1.6180, 0.0073]])
tensor([ 2,  3,  5,  7, 11, 13, 17, 19])


### 2.2 Factory functions
Most of the time you ask for a tensor of a given shape. The names are the same as in NumPy.

In [7]:
print(torch.zeros(2, 3))
print(torch.ones(2, 3))
print(torch.full((2, 3), 7.0))
print(torch.eye(3))                          # identity matrix

tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[7., 7., 7.],
        [7., 7., 7.]])
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])


In [8]:
print(torch.arange(0, 10))                   # 0, 1, ..., 9 (the end is excluded, like range)
print(torch.arange(0, 10, 2))                # step 2
print(torch.linspace(0, 1, 5))               # 5 evenly spaced numbers from 0 to 1

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
tensor([0, 2, 4, 6, 8])
tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


**Random tensors** are how neural networks start: the weights begin with random values, and training improves them.

In [9]:
print(torch.rand(2, 3))                      # uniform on [0, 1)
print(torch.randn(2, 3))                     # standard normal: mean 0, std 1
print(torch.randint(0, 10, (2, 3)))          # random integers from 0 to 9

tensor([[0.5209, 0.0356, 0.0521],
        [0.2301, 0.8080, 0.4545]])
tensor([[ 1.1877, -1.6208, -0.8955],
        [ 1.3849, -0.7163,  2.3047]])
tensor([[7, 4, 1],
        [8, 8, 4]])


In [10]:
random_image = torch.rand(3, 224, 224)       # a random "image": 3 color channels, 224 × 224 pixels
print(random_image.shape, random_image.ndim)

torch.Size([3, 224, 224]) 3


### 2.3 Same shape as another tensor: `*_like`

*Adapted from the PyTorch tutorial [Introduction to PyTorch Tensors](https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html), BSD 3-Clause License (modified).*

In [11]:
x = torch.rand(2, 2, 3)
print(torch.zeros_like(x).shape)
print(torch.ones_like(x).shape)
print(torch.rand_like(x).shape)

torch.Size([2, 2, 3])
torch.Size([2, 2, 3])
torch.Size([2, 2, 3])


## 3 · Data types

Every tensor has **one** data type (`dtype`) for all its numbers. The defaults:
- integers → `torch.int64`
- decimals → **`torch.float32`**, the type neural networks use

In [12]:
print(torch.tensor([1, 2, 3]).dtype)
print(torch.tensor([1.0, 2.0, 3.0]).dtype)

torch.int64
torch.float32


Choose the type at creation with `dtype=`, or convert later with `.to()` (or the shortcuts `.float()`, `.long()`, `.int()`):

In [13]:
a = torch.ones((2, 3), dtype=torch.int16)
print(a)

b = torch.rand((2, 3), dtype=torch.float64) * 20.0
print(b)

c = b.to(torch.int32)           # converting to integers truncates the decimals
print(c)
print(b.float().dtype, c.long().dtype)

tensor([[1, 1, 1],
        [1, 1, 1]], dtype=torch.int16)
tensor([[ 7.2411,  6.2582, 12.5984],
        [18.9774, 17.0906,  5.8181]], dtype=torch.float64)
tensor([[ 7,  6, 12],
        [18, 17,  5]], dtype=torch.int32)
torch.float32 torch.int64


**The float64 trap.** NumPy's default for decimals is `float64`, and PyTorch **keeps** that type when it converts a NumPy array. A model expects `float32`, so you get errors like *"expected scalar type Float but found Double"*. Convert with `.float()`.

In [14]:
arr = np.array([1.0, 2.0, 3.0])
t = torch.tensor(arr)
print(arr.dtype, "->", t.dtype)
print(t.float().dtype)

float64 -> torch.float64
torch.float32


Some operations need a floating-point type. For example, `mean` refuses integers:

In [15]:
ints = torch.arange(0, 100, 10)
try:
    ints.mean()
except RuntimeError as e:
    print("RuntimeError:", e)
print(ints.float().mean())

RuntimeError: mean(): could not infer output dtype. Input dtype must be either a floating point or complex dtype. Got: Long
tensor(45.)


**Three questions to ask when something goes wrong:** what is the tensor's **shape**? its **dtype**? its **device**? These three attributes cause most errors in deep learning code.

In [16]:
some_tensor = torch.rand(3, 4)
print(f"shape: {some_tensor.shape} | dtype: {some_tensor.dtype} | device: {some_tensor.device}")

shape: torch.Size([3, 4]) | dtype: torch.float32 | device: cpu


## 4 · Operations

### 4.1 Element-wise arithmetic
With a number (a scalar), the operation is applied to every element. Between two tensors of the **same shape**, it is applied element by element.

*Adapted from the PyTorch tutorial [Introduction to PyTorch Tensors](https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html), BSD 3-Clause License (modified).*

In [17]:
ones = torch.zeros(2, 2) + 1
twos = torch.ones(2, 2) * 2
threes = (torch.ones(2, 2) * 7 - 1) / 2
fours = twos ** 2
print(ones, twos, threes, fours, sep="\n")

tensor([[1., 1.],
        [1., 1.]])
tensor([[2., 2.],
        [2., 2.]])
tensor([[3., 3.],
        [3., 3.]])
tensor([[4., 4.],
        [4., 4.]])


In [18]:
powers2 = twos ** torch.tensor([[1, 2], [3, 4]])
print(powers2)
print(threes * fours)                        # element-wise product

tensor([[ 2.,  4.],
        [ 8., 16.]])
tensor([[12., 12.],
        [12., 12.]])


In [19]:
x = torch.tensor([-2.0, 0.0, 2.0])
print(torch.exp(x), torch.abs(x), torch.clamp(x, -1, 1), sep="\n")
print(1 / (1 + torch.exp(-x)))              # the sigmoid function, used from week 2

tensor([0.1353, 1.0000, 7.3891])
tensor([2., 0., 2.])
tensor([-1.,  0.,  1.])
tensor([0.1192, 0.5000, 0.8808])


### 4.2 In-place operations
`a + b` creates a **new** tensor. Methods whose name ends with an underscore `_` change the tensor **in place**. You will see this convention all semester (for example `zero_()` in week 2).

*Adapted from the PyTorch tutorial [Introduction to PyTorch Tensors](https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html), BSD 3-Clause License (modified).*

In [20]:
a = torch.ones(2, 2)
b = torch.rand(2, 2)
a.add_(b)                                    # a is changed; b is not
print(a)
print(b)

tensor([[1.3846, 1.4903],
        [1.1869, 1.0079]])
tensor([[0.3846, 0.4903],
        [0.1869, 0.0079]])


### 4.3 Matrix multiplication
Two different products:
- `*`: element-wise, the shapes must match;
- `@` (or `torch.matmul`): the **matrix product**.

The matrix product follows two rules:
1. The **inner** dimensions must match: `(3, 2) @ (2, 4)` works, `(3, 2) @ (3, 2)` does not.
2. The result has the **outer** dimensions: `(3, 2) @ (2, 4)` → `(3, 4)`.

*Adapted from D. Bourke, [Learn PyTorch for Deep Learning: 00. PyTorch Fundamentals](https://www.learnpytorch.io/00_pytorch_fundamentals/), MIT License (modified).*

In [21]:
t = torch.tensor([1, 2, 3])
print(t * t)                                 # element-wise: [1, 4, 9]
print(t @ t)                                 # dot product: 1 + 4 + 9 = 14

tensor([1, 4, 9])
tensor(14)


In [22]:
A = torch.tensor([[1, 2], [3, 4], [5, 6]], dtype=torch.float32)      # (3, 2)
B = torch.tensor([[7, 10], [8, 11], [9, 12]], dtype=torch.float32)   # (3, 2)
try:
    A @ B
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (3x2 and 3x2)


This is **the most common error in deep learning**: a shape mismatch. Read the message, it gives both shapes. Here the fix is to **transpose** one matrix: `B.T` swaps its rows and columns, so `(3, 2)` becomes `(2, 3)`.

In [23]:
print(A.shape, "@", B.T.shape)
print(A @ B.T)
print("result shape:", (A @ B.T).shape)

torch.Size([3, 2]) @ torch.Size([2, 3])
tensor([[ 27.,  30.,  33.],
        [ 61.,  68.,  75.],
        [ 95., 106., 117.]])
result shape: torch.Size([3, 3])


**Why it matters for this course.** A linear layer computes, for all m examples at once,
$$Z = X W + b, \qquad X: (m, n), \quad W: (n, k), \quad b: (k,), \quad Z: (m, k).$$

In [24]:
m, n, k = 5, 3, 2
X = torch.rand(m, n)                         # 5 examples, 3 features each
W = torch.rand(n, k)
b = torch.rand(k)
Z = X @ W + b
print("Z shape:", Z.shape)                   # one row of k outputs per example

Z shape: torch.Size([5, 2])


Matrix multiplication is also **much faster** than doing the same work with Python loops:

In [25]:
import time

def matmul_loops(A, B):
    out = torch.zeros(A.shape[0], B.shape[1])
    for i in range(A.shape[0]):
        for j in range(B.shape[1]):
            for k in range(A.shape[1]):
                out[i, j] += A[i, k] * B[k, j]
    return out

A, B = torch.rand(30, 30), torch.rand(30, 30)
start = time.perf_counter(); C1 = matmul_loops(A, B); t_loops = time.perf_counter() - start
start = time.perf_counter(); C2 = A @ B;               t_mm = time.perf_counter() - start
print(f"loops: {t_loops * 1000:.1f} ms | @: {t_mm * 1000:.3f} ms | same result: {torch.allclose(C1, C2)}")

loops: 122.8 ms | @: 6.009 ms | same result: True


## 5 · Aggregation: min, max, mean, sum

*Adapted from D. Bourke, [Learn PyTorch for Deep Learning: 00. PyTorch Fundamentals](https://www.learnpytorch.io/00_pytorch_fundamentals/), MIT License (modified).*

In [26]:
x = torch.arange(0, 100, 10).float()
print(x)
print("min:", x.min(), "| max:", x.max(), "| mean:", x.mean(), "| sum:", x.sum())
print("argmin:", x.argmin(), "| argmax:", x.argmax())      # positions, not values

tensor([ 0., 10., 20., 30., 40., 50., 60., 70., 80., 90.])
min: tensor(0.) | max: tensor(90.) | mean: tensor(45.) | sum: tensor(450.)
argmin: tensor(0) | argmax: tensor(9)


On a matrix, `dim` says **which dimension disappears**:
- `dim=0` collapses the rows → one value **per column** (per feature);
- `dim=1` collapses the columns → one value **per row** (per example).

`keepdim=True` keeps the collapsed dimension with size 1, which is useful for broadcasting (Part 8).

In [27]:
M = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(M.sum(dim=0))                          # per column: [5, 7, 9]
print(M.sum(dim=1))                          # per row: [6, 15]
print(M.sum(dim=1, keepdim=True).shape)      # (2, 1)
print(M.max(dim=1))                          # values and positions

tensor([5., 7., 9.])
tensor([ 6., 15.])
torch.Size([2, 1])
torch.return_types.max(
values=tensor([3., 6.]),
indices=tensor([2, 2]))


From week 3, a network outputs one score per class for every example, shape (m, number of classes). The predicted class is `argmax(dim=1)`:

In [28]:
scores = torch.tensor([[0.1, 2.5, 0.3],
                       [1.2, 0.2, 0.9]])     # 2 examples, 3 classes
print("predicted classes:", scores.argmax(dim=1))

predicted classes: tensor([1, 0])


## 6 · Reshaping, stacking, squeezing, permuting

| Method | What it does |
|---|---|
| `x.reshape(shape)` | new shape, same values in the same order (the number of elements cannot change) |
| `x.view(shape)` | like `reshape`, but always shares memory with `x` |
| `torch.stack`, `torch.cat` | combine tensors: along a **new** dimension, or an **existing** one |
| `x.squeeze()` | remove dimensions of size 1 |
| `x.unsqueeze(d)` | add a dimension of size 1 at position `d` |
| `x.permute(...)` | reorder the dimensions |

*Adapted from D. Bourke, [Learn PyTorch for Deep Learning: 00. PyTorch Fundamentals](https://www.learnpytorch.io/00_pytorch_fundamentals/), MIT License (modified).*

In [29]:
x = torch.arange(1.0, 10.0)
print(x, x.shape)
print(x.reshape(3, 3))
print(x.reshape(1, 9).shape, x.reshape(9, 1).shape)
print(x.reshape(3, -1).shape)                # -1: computed for you (9 / 3 = 3)

tensor([1., 2., 3., 4., 5., 6., 7., 8., 9.]) torch.Size([9])
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
torch.Size([1, 9]) torch.Size([9, 1])
torch.Size([3, 3])


In [30]:
z = x.view(3, 3)
z[0, 0] = 100                                # changing the view ...
print(x)                                     # ... changes x too: they share memory

tensor([100.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.])


In [31]:
a, b = torch.ones(3), torch.zeros(3)
print(torch.cat([a, b]))                     # (6,)
print(torch.stack([a, b]))                   # (2, 3): new dimension in front
print(torch.stack([a, b], dim=1))            # (3, 2)

tensor([1., 1., 1., 0., 0., 0.])
tensor([[1., 1., 1.],
        [0., 0., 0.]])
tensor([[1., 0.],
        [1., 0.],
        [1., 0.]])


**`unsqueeze` for a batch of one.** Models expect a **batch**: shape (N, C, H, W). To give them a single image, add a batch dimension of size 1.

*Adapted from the PyTorch tutorial [Introduction to PyTorch Tensors](https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html), BSD 3-Clause License (modified).*

In [32]:
img = torch.rand(3, 226, 226)
batch = img.unsqueeze(0)
print(img.shape, "->", batch.shape)

out = torch.rand(1, 20)                      # a model's output for a batch of one
print(out.shape, "->", out.squeeze(0).shape)
print(torch.rand(1, 1, 1, 10).squeeze().shape)

torch.Size([3, 226, 226]) -> torch.Size([1, 3, 226, 226])
torch.Size([1, 20]) -> torch.Size([20])
torch.Size([10])


**`permute` for images.** Image files, NumPy and Matplotlib store images as **(H, W, C)**; PyTorch layers expect **(C, H, W)**. `permute` reorders the dimensions. Using `reshape` instead gives the right **shape** but **scrambled pixels**.

In [33]:
img_hwc = torch.rand(32, 32, 3)              # height, width, color
img_chw = img_hwc.permute(2, 0, 1)           # new order: old dim 2, then 0, then 1
print(img_hwc.shape, "->", img_chw.shape)
print("same pixel:", img_hwc[5, 7, 2].item(), img_chw[2, 5, 7].item())

wrong = img_hwc.reshape(3, 32, 32)
print("reshape gives the same image?", torch.equal(wrong, img_chw))

torch.Size([32, 32, 3]) -> torch.Size([3, 32, 32])
same pixel: 0.2709875702857971 0.2709875702857971
reshape gives the same image? False


## 7 · Indexing
Exactly like NumPy. For a matrix, **the first index is the row**.

*Adapted from D. Bourke, [Learn PyTorch for Deep Learning: 00. PyTorch Fundamentals](https://www.learnpytorch.io/00_pytorch_fundamentals/), MIT License (modified).*

In [34]:
x = torch.arange(1, 10).reshape(1, 3, 3)
print(x)
print(x[0])                                  # the first (and only) block
print(x[0, 0])                               # its first row
print(x[0, 2, 2])                            # one number
print(x[:, :, 1])                            # the middle column, all rows
print(x[x > 5])                              # boolean mask: all values greater than 5

tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])
tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])
tensor([1, 2, 3])
tensor(9)
tensor([[2, 5, 8]])
tensor([6, 7, 8, 9])


## 8 · Broadcasting

Operations on tensors of **different** shapes are sometimes allowed: dimensions of size 1 are stretched to match, without copying any data. This is called **broadcasting**. You may know it from NumPy; **PyTorch uses exactly the same rules**.

**The rule.** Compare the shapes **from the right**. Two dimensions are compatible if they are **equal**, or **one of them is 1**, or **one of them is missing**.

### 8.1 In NumPy

In [35]:
X = np.zeros((5, 3))
print((X + np.array([1.0, 2.0, 3.0])).shape)            # (5, 3) + (3,): added to every row
print((X + np.arange(5.0).reshape(5, 1)).shape)         # (5, 3) + (5, 1): added to every column

col = np.arange(1.0, 4.0).reshape(3, 1)                 # (3, 1)
row = np.arange(1.0, 5.0).reshape(1, 4)                 # (1, 4)
print(col * row)                                        # (3, 4): a multiplication table

(5, 3)
(5, 3)
[[ 1.  2.  3.  4.]
 [ 2.  4.  6.  8.]
 [ 3.  6.  9. 12.]]


### 8.2 The same in PyTorch

In [36]:
table = torch.from_numpy(col) * torch.from_numpy(row)
print(table)
print("same as NumPy:", np.array_equal(table.numpy(), col * row))

tensor([[ 1.,  2.,  3.,  4.],
        [ 2.,  4.,  6.,  8.],
        [ 3.,  6.,  9., 12.]], dtype=torch.float64)
same as NumPy: True


In [37]:
a = torch.ones(4, 3, 2)
b = a * torch.rand(3, 2)       # last two dims identical, first one missing
c = a * torch.rand(3, 1)       # last dim is 1, middle dim identical
d = a * torch.rand(1, 2)       # last dim identical, middle dim is 1
print(b.shape, c.shape, d.shape)
print(c[0])                    # every row of the 3 × 2 block is multiplied by its own number

torch.Size([4, 3, 2]) torch.Size([4, 3, 2]) torch.Size([4, 3, 2])
tensor([[0.7322, 0.7322],
        [0.6541, 0.6541],
        [0.7344, 0.7344]])


When the rule is not satisfied, both libraries raise an error:

In [38]:
try:
    np.ones((4, 3, 2)) * np.ones((4, 3))
except ValueError as e:
    print("NumPy ValueError:", e)
try:
    torch.ones(4, 3, 2) * torch.ones(4, 3)
except RuntimeError as e:
    print("PyTorch RuntimeError:", e)

NumPy ValueError: operands could not be broadcast together with shapes (4,3,2) (4,3) 
PyTorch RuntimeError: The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 2


Broadcasting is how we used it in Part 4.3: in `X @ W + b`, the bias `b` of shape (k,) is added to every row of the (m, k) result. And with `keepdim=True` (Part 5), per-column statistics broadcast back over the rows:

In [39]:
data = torch.rand(100, 4) * 10
centered = data - data.mean(dim=0, keepdim=True)        # (100, 4) - (1, 4)
print(centered.mean(dim=0))                             # every column now has mean ≈ 0

tensor([ 2.4080e-07, -2.0981e-07, -4.1962e-07,  1.0729e-07])


### 8.3 A wrong result without an error
Broadcasting errors are good: they stop you. The dangerous case is when broadcasting **succeeds and gives the wrong answer**. Predictions often have shape (m, 1), because that is what `X @ W` returns; labels often have shape (m,), because that is how datasets store them:

In [40]:
y_hat = torch.tensor([[1.0], [2.0], [3.0], [4.0]])     # (4, 1)
y = torch.tensor([1.0, 2.0, 3.0, 4.0])                 # (4,)

diff = y_hat - y
print("diff shape:", diff.shape)                       # (4, 4), not (4,)!
print("mean squared error:", (diff ** 2).mean().item())  # should be 0: the predictions are perfect
print("NumPy does the same:", (y_hat.numpy() - y.numpy()).shape)

diff shape: torch.Size([4, 4])
mean squared error: 2.5
NumPy does the same: (4, 4)


The predictions are perfect, but the error is 2.5, and there is **no error message**. The fix: make both tensors the same shape first.

In [41]:
diff = y_hat.squeeze(1) - y                  # or: y_hat - y.reshape(-1, 1)
print(diff.shape, (diff ** 2).mean().item())

torch.Size([4]) 0.0


**Habit for the whole semester:** print the shapes when you write a new computation, and make sure they are what you expect.

## 9 · Copying tensors, and PyTorch ↔ NumPy

### 9.1 Assignment does not copy
`b = a` gives the same tensor a second name. To get an independent copy, use `clone()` (NumPy: `copy()`).

*Adapted from the PyTorch tutorial [Introduction to PyTorch Tensors](https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html), BSD 3-Clause License (modified).*

In [42]:
a = torch.ones(2, 2)
b = a
a[0, 1] = 561
print(b)                                     # b changed too: same tensor

a = torch.ones(2, 2)
b = a.clone()
a[0, 1] = 561
print(b)                                     # b is unchanged

tensor([[  1., 561.],
        [  1.,   1.]])
tensor([[1., 1.],
        [1., 1.]])


### 9.2 NumPy ↔ PyTorch
| Direction | Function | Memory |
|---|---|---|
| NumPy → PyTorch | `torch.from_numpy(arr)` | **shared** with `arr` |
| NumPy → PyTorch | `torch.tensor(arr)` | copied |
| PyTorch → NumPy | `t.numpy()` | **shared** with `t` (CPU tensors only) |

*Adapted from the PyTorch tutorial [Introduction to PyTorch Tensors](https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html), BSD 3-Clause License (modified).*

In [43]:
numpy_array = np.ones((2, 3))
pytorch_tensor = torch.from_numpy(numpy_array)
numpy_array[1, 1] = 23
print(pytorch_tensor)                        # changed too, and dtype float64 (NumPy's default)

copied = torch.tensor(numpy_array)
numpy_array[0, 0] = -1
print(copied)                                # not changed

tensor([[ 1.,  1.,  1.],
        [ 1., 23.,  1.]], dtype=torch.float64)
tensor([[ 1.,  1.,  1.],
        [ 1., 23.,  1.]], dtype=torch.float64)


## 10 · Reproducibility
Random numbers make every run different. When you want the same "random" numbers again, for example to compare two experiments fairly, set the **seed** of the random number generator.

*Adapted from the PyTorch tutorial [Introduction to PyTorch Tensors](https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html), BSD 3-Clause License (modified).*

In [44]:
torch.manual_seed(1729)
random1 = torch.rand(2, 3)
random2 = torch.rand(2, 3)

torch.manual_seed(1729)
random3 = torch.rand(2, 3)
random4 = torch.rand(2, 3)

print(torch.equal(random1, random3), torch.equal(random2, random4))   # True True
print(torch.equal(random1, random2))                                    # False

True True
False


Setting the seed **resets** the generator: the sequence of numbers after it is always the same. Note that `random2` is different from `random1`: the seed does not repeat the same number, it repeats the same **sequence**. So the seed only helps if the code between the seed and your tensor is exactly the same.

## 11 · Running on the GPU

A GPU has thousands of small cores and its **own memory**. To compute on it, the data must be **moved** to the GPU's memory.

### 11.1 Device-agnostic code
Write code that works with and without a GPU: choose the device once, then use it everywhere.

*Adapted from D. Bourke, [Learn PyTorch for Deep Learning: 00. PyTorch Fundamentals](https://www.learnpytorch.io/00_pytorch_fundamentals/), MIT License (modified).*

In [45]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

x = torch.tensor([1.0, 2.0, 3.0])            # created on the CPU
x_dev = x.to(device)                         # a copy on the device
print(x.device, "->", x_dev.device)

y_dev = torch.rand(3, device=device)         # or create it there directly

device: cpu
cpu -> cpu


### 11.2 All tensors must be on the same device

In [46]:
if device == "cuda":
    try:
        x + x_dev                            # one on the CPU, one on the GPU
    except RuntimeError as e:
        print("RuntimeError:", str(e)[:120], "...")
else:
    print("No GPU in this runtime: this error only happens on a GPU.")

No GPU in this runtime: this error only happens on a GPU.


### 11.3 Back to the CPU
NumPy only works on the CPU: bring the tensor back first with `.cpu()`.

In [47]:
try:
    x_dev.numpy()
    print("On the CPU, .numpy() works directly.")
except TypeError as e:
    print("TypeError:", str(e)[:100], "...")
print(x_dev.cpu().numpy())

On the CPU, .numpy() works directly.
[1. 2. 3.]


### 11.4 How much faster?
Two details for fair timing on a GPU: run once before measuring (**warm-up**), and call `torch.cuda.synchronize()`, because GPU operations return before the work is finished.

In [48]:
def time_matmul(n, device, repeats=5):
    A = torch.randn(n, n, device=device)
    B = torch.randn(n, n, device=device)
    A @ B                                    # warm-up
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        A @ B
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / repeats

for n in [256, 2048]:
    t_cpu = time_matmul(n, "cpu")
    line = f"n = {n:5d} | CPU {t_cpu * 1000:8.2f} ms"
    if device == "cuda":
        t_gpu = time_matmul(n, "cuda")
        line += f" | GPU {t_gpu * 1000:7.2f} ms | {t_cpu / t_gpu:6.1f}x faster"
    print(line)

n =   256 | CPU     0.03 ms
n =  2048 | CPU    11.60 ms


Training a network is mostly large matrix products like these, which is why deep learning runs on GPUs. For small matrices the gain is smaller: moving work to the GPU has a fixed cost.

---
# ★ Exercises

Do them **on your own**, in order, in the empty cells. Use the documentation. Where an expected output is shown, your output should match it (on a GPU, tensors also show `device='cuda:0'`).

## A · Warm-up

### Exercise 1
Create a random tensor with shape `(7, 7)`, then multiply it (matrix product) by another random tensor with shape `(1, 7)`. *Hint:* you will need to transpose the second tensor. Print the result and its shape.

In [49]:
# Create two random tensors
A = torch.rand(7, 7)
B = torch.rand(1, 7)

# Matrix multiplication (check the shapes first)
result = A @ B.T
print(result, result.shape)

tensor([[1.9281],
        [1.2306],
        [0.4578],
        [1.0589],
        [0.8832],
        [1.3486],
        [0.6516]]) torch.Size([7, 1])


### Exercise 2
Set the random seed to `0` and do Exercise 1 again. Print the result and its shape. Expected output:
```
tensor([[1.8542],
        [1.9611],
        [2.2884],
        [3.0481],
        [1.7067],
        [2.5290],
        [1.7989]]) torch.Size([7, 1])
```

In [50]:
# Set the seed
torch.manual_seed(0)

# Create two random tensors
A = torch.rand(7, 7)
B = torch.rand(1, 7)

# Matrix multiplication
result = A @ B.T
print(result, result.shape)

tensor([[1.8542],
        [1.9611],
        [2.2884],
        [3.0481],
        [1.7067],
        [2.5290],
        [1.7989]]) torch.Size([7, 1])


### Exercise 3
Is there a random seed for the **GPU** too? Find it in the documentation of [`torch.cuda`](https://pytorch.org/docs/stable/cuda.html) and set it to `1234`. In a comment, explain in one sentence why `torch.manual_seed` is usually enough.

In [51]:
# Set the GPU seed
torch.cuda.manual_seed(1234)

# Why is torch.manual_seed usually enough?
# It seeds the random number generators on all devices.

### Exercise 4
Set `torch.manual_seed(1234)`, create two random tensors of shape `(2, 3)` **on the CPU**, then move both to `device`. Multiply them (matrix product; adjust the shape of one of them). Print the result and its shape. Expected output (on a CPU):
```
tensor([[0.3647, 0.4709],
        [0.5184, 0.5617]]) torch.Size([2, 2])
```
*Why create them on the CPU first?* The GPU has its own random number generator: the same seed gives different numbers on the GPU.

In [52]:
# Set the seed
torch.manual_seed(1234)

# Create two random tensors on the CPU and move them to the device
A = torch.rand(2, 3, device="cpu").to(device)
B = torch.rand(2, 3, device="cpu").to(device)

# Matrix multiplication
result = A @ B.T
print(result, result.shape)

tensor([[0.3647, 0.4709],
        [0.5184, 0.5617]]) torch.Size([2, 2])


### Exercise 5
Find the maximum and minimum **values** of the output of Exercise 4, then the **positions** of the maximum and minimum (`argmax`, `argmin`). Expected output:
```
tensor(0.5617) tensor(0.3647)
tensor(3) tensor(0)
```

In [53]:
# Max and min
print(result.max(), result.min())

# Positions of the max and min
print(result.argmax(), result.argmin())

tensor(0.5617) tensor(0.3647)
tensor(3) tensor(0)


### Exercise 6
Set the seed to `7`, create a random tensor of shape `(1, 1, 1, 10)`, then a new tensor with all the size-1 dimensions removed, of shape `(10,)`. Print both tensors and their shapes. Expected output:
```
tensor([[[[0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297,
           0.3653, 0.8513]]]]) torch.Size([1, 1, 1, 10])
tensor([0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297, 0.3653,
        0.8513]) torch.Size([10])
```

In [54]:
# Set the seed
torch.manual_seed(7)

# Create a random tensor
x = torch.rand(1, 1, 1, 10)

# Remove the size-1 dimensions
y = x.squeeze()

# Print both tensors and their shapes
print(x, x.shape)
print(y, y.shape)

tensor([[[[0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297,
           0.3653, 0.8513]]]]) torch.Size([1, 1, 1, 10])
tensor([0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297, 0.3653,
        0.8513]) torch.Size([10])


### Exercise 7
The NumPy array below holds the heights of five students, in meters. Convert it to a PyTorch tensor with dtype **`float32`**, then print the tensor's dtype and the mean height, rounded to 3 decimals. Expected output:
```
torch.float32
1.688
```

In [55]:
heights = np.array([1.62, 1.75, 1.58, 1.80, 1.69])

# Convert to a float32 tensor
heights_tensor = torch.tensor(heights, dtype=torch.float32)

# Print its dtype and its mean (3 decimals)
print(heights_tensor.dtype)
print(round(heights_tensor.mean().item(), 3))

torch.float32
1.688


## B · Applied exercises

### Exercise 8 · From images to a data matrix
`batch` holds 8 RGB images of 32 × 32 pixels in **(N, H, W, C)** format, as an image library would give them.
1. Convert it to PyTorch's format **(N, C, H, W)**: `batch_nchw`.
2. Flatten each image into one row, to get a data matrix `X_images` of shape **(m, n) = (8, 3072)**.
3. Check that pixel (row 10, column 20, green channel) of image 4 is the same in `batch` and `batch_nchw`.

Expected output:
```
torch.Size([8, 3, 32, 32]) torch.Size([8, 3072])
tensor(True)
```

In [56]:
torch.manual_seed(0)
batch = torch.rand(8, 32, 32, 3)

# 1. (N, H, W, C) -> (N, C, H, W)
batch_nchw = batch.permute(0, 3, 1, 2)

# 2. One row per image
X_images = batch_nchw.reshape(8, -1)

# 3. Compare one pixel (green is channel 1)
print(batch_nchw.shape, X_images.shape)
print(torch.equal(batch[4, 10, 20, 1], batch_nchw[4, 1, 10, 20]))

torch.Size([8, 3, 32, 32]) torch.Size([8, 3072])
True


### Exercise 9 · Predict the broadcast shapes
**Without running code**, write in the comments the shape of `A + B` for each pair, or "error". Then check your answers with NumPy: `np.broadcast_shapes(shape_a, shape_b)` returns the result shape, or raises a `ValueError`.

| | shape of A | shape of B |
|---|---|---|
| a | (4, 1) | (3,) |
| b | (2, 3) | (2, 1) |
| c | (2, 3) | (2,) |
| d | (8, 1, 5) | (7, 1) |
| e | (10, 1) | (10,) |

Expected output of the check:
```
a (4, 3)
b (2, 3)
c error
d (8, 7, 5)
e (10, 10)
```

In [57]:
# My predictions:
# a: (4, 3)
# b: (2, 3)
# c: error
# d: (8, 7, 5)
# e: (10, 10)

# Check with np.broadcast_shapes
shapes = {
    "a": ((4, 1), (3,)),
    "b": ((2, 3), (2, 1)),
    "c": ((2, 3), (2,)),
    "d": ((8, 1, 5), (7, 1)),
    "e": ((10, 1), (10,))
}
for name, (shape_a, shape_b) in shapes.items():
    try:
        print(name, np.broadcast_shapes(shape_a, shape_b))
    except ValueError:
        print(name, "error")

a (4, 3)
b (2, 3)
c error
d (8, 7, 5)
e (10, 10)


### Exercise 10 · Standardize the features
Each column of `data` is one feature, with a very different scale. Standardize every column: subtract its mean and divide by its standard deviation, **without loops**. Compute `mu` and `sigma` of shape (1, 4) with `keepdim=True`. Then print the mean and the standard deviation of each column of the result, rounded to 4 decimals. Expected output:
```
torch.Size([1, 4]) torch.Size([1, 4])
tensor([0., 0., 0., 0.])
tensor([1., 1., 1., 1.])
```

In [58]:
torch.manual_seed(0)
data = torch.randn(100, 4) * torch.tensor([1.0, 10.0, 100.0, 0.1]) + 5

# Per-column mean and standard deviation, shape (1, 4)
mu = data.mean(dim=0, keepdim=True)
sigma = data.std(dim=0, keepdim=True)

# Standardize
standardized = (data - mu) / sigma

# Print the shapes, then the mean and std of each column (4 decimals)
# (use .abs() on the means, so that -0.0 prints as 0.0)
print(mu.shape, sigma.shape)
print(standardized.mean(dim=0).abs().round(decimals=4))
print(standardized.std(dim=0).round(decimals=4))

torch.Size([1, 4]) torch.Size([1, 4])
tensor([0., 0., 0., 0.])
tensor([1., 1., 1., 1.])


### Exercise 11 · Find the bug
This function should compute the mean squared error between predictions and labels. For the inputs below it returns a wrong value, without any error. Explain the bug in a comment, fix the function by changing **one line**, and add an `assert` that checks that the two shapes are equal. Expected output after the fix:
```
1.6667
```

In [59]:
# The bug: y_hat has shape (3, 1) while y has shape (3,), so broadcasting
# creates a (3, 3) matrix instead of comparing matching prediction/label pairs.
def mse(y_hat, y):
    assert y_hat.shape == y.shape
    return ((y_hat - y) ** 2).mean()

y_hat = torch.tensor([[2.0], [0.0], [1.0]])
y = torch.tensor([1.0, 0.0, 3.0]).reshape_as(y_hat)
print(round(mse(y_hat, y).item(), 4))

1.6667


### Exercise 12 · Replace the loops
Each function below uses Python loops. Write the **vectorized** version (tensor operations only, no `for`), then check that both give the same result with `torch.allclose`, and compare their speed with `time.perf_counter()`.

**(a)** A linear model for every example: $z_i = \sum_j X_{ij} w_j + b$. This is the model of next week.

**(b)** Min-max scaling of every **column** to [0, 1]: $X'_{ij} = (X_{ij} - \min_k X_{kj}) / (\max_k X_{kj} - \min_k X_{kj})$. *Hint:* `amin` and `amax` with `dim` and `keepdim`.

Expected output (your times will differ):
```
linear: same result True | loops ... ms, vectorized ... ms
min-max: same result True | loops ... ms, vectorized ... ms
```

In [60]:
def linear_loop(X, w, b):
    z = torch.zeros(X.shape[0])
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            z[i] += X[i, j] * w[j]
        z[i] += b
    return z

def minmax_loop(X):
    out = torch.empty_like(X)
    for j in range(X.shape[1]):
        lo, hi = X[:, j].min(), X[:, j].max()
        for i in range(X.shape[0]):
            out[i, j] = (X[i, j] - lo) / (hi - lo)
    return out

def linear_vec(X, w, b):
    return X @ w + b

def minmax_vec(X):
    lo = X.amin(dim=0, keepdim=True)
    hi = X.amax(dim=0, keepdim=True)
    return (X - lo) / (hi - lo)

torch.manual_seed(0)
X = torch.randn(200, 30)
w = torch.randn(30)
b = 0.5

# Compare results (torch.allclose) and times (time.perf_counter) for both pairs
import time

start = time.perf_counter()
z_loop = linear_loop(X, w, b)
loop_ms = (time.perf_counter() - start) * 1000
start = time.perf_counter()
z_vec = linear_vec(X, w, b)
vec_ms = (time.perf_counter() - start) * 1000
print(f"linear: same result {torch.allclose(z_loop, z_vec)} | loops {loop_ms:.3f} ms, vectorized {vec_ms:.3f} ms")

start = time.perf_counter()
scaled_loop = minmax_loop(X)
loop_ms = (time.perf_counter() - start) * 1000
start = time.perf_counter()
scaled_vec = minmax_vec(X)
vec_ms = (time.perf_counter() - start) * 1000
print(f"min-max: same result {torch.allclose(scaled_loop, scaled_vec)} | loops {loop_ms:.3f} ms, vectorized {vec_ms:.3f} ms")

linear: same result False | loops 19.822 ms, vectorized 1.829 ms
min-max: same result True | loops 19.502 ms, vectorized 0.100 ms


---
## Before you submit
1. **Runtime → Restart session and run all**, and check that every exercise runs without errors.
2. **File → Download → Download .ipynb**, name it `Notebook1_<StudentID>.ipynb`, and submit it before the end of the session.

**Extra reading, if you want more:** the PyTorch tutorial [Introduction to PyTorch Tensors](https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html) and [00. PyTorch Fundamentals](https://www.learnpytorch.io/00_pytorch_fundamentals/) by Daniel Bourke, which this notebook follows.

**Next week:** logistic regression as a neural network. You will write the model `X @ w + b`, the sigmoid and the cost with the operations from today.

---
*Credits: Parts of this notebook are adapted from the PyTorch tutorial [Introduction to PyTorch Tensors](https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html) (PyTorch contributors, BSD 3-Clause License) and from D. Bourke, [Learn PyTorch for Deep Learning: 00. PyTorch Fundamentals](https://github.com/mrdbourke/pytorch-deep-learning) (MIT License), including Exercises 1–6. The code and text were modified.*